# ORBIT R3M Validation on LeRobot Datasets

This notebook validates ORBIT's R3M-based profiler on real LeRobot datasets.

**Requirements:** Run on Google Colab with a T4 GPU (Runtime > Change runtime type > T4).
Expected runtime: ~20-30 minutes.

In [ ]:
# Cell 1 — Setup
!pip install -q orbit-robotics[r3m,profile,lerobot]

import torch
assert torch.cuda.is_available(), (
    "GPU not available! Go to Runtime > Change runtime type > T4 GPU"
)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"CUDA version: {torch.version.cuda}")
print(f"PyTorch version: {torch.__version__}")

In [ ]:
# Cell 2 — Profile LeRobot datasets with R3M
import time
from orbit.profile.profiler import DatasetProfiler
from orbit.profile.report_card import ReportCardGenerator

profiler = DatasetProfiler(embedding_model="r3m", device="cuda")
report_gen = ReportCardGenerator()

datasets = {
    "pusht": {
        "repo": "lerobot/pusht",
        "tasks": ["push block to target", "precise positioning"],
    },
    "aloha_transfer": {
        "repo": "lerobot/aloha_sim_transfer_cube_human",
        "tasks": ["pick up cube", "bimanual handover", "place cube"],
    },
    "aloha_insertion": {
        "repo": "lerobot/aloha_sim_insertion_human",
        "tasks": ["align peg", "insert peg"],
    },
    "xarm_lift": {
        "repo": "lerobot/xarm_lift_medium_replay",
        "tasks": ["grasp object", "lift object"],
    },
}

results = {}
for name, config in datasets.items():
    print(f"\n{'='*60}")
    print(f"Profiling {name} ({config['repo']})...")
    start = time.time()
    
    profile = profiler.profile_from_hub(
        config["repo"],
        task_descriptions=config["tasks"],
        max_episodes=50,
    )
    
    elapsed = time.time() - start
    card = report_gen.generate(profile)
    results[name] = {
        "profile": profile,
        "report_card": card,
        "time_seconds": elapsed,
    }
    
    # Print report card
    print(report_gen.render_cli(card))
    print(f"Profiled in {elapsed:.1f}s")

In [ ]:
# Cell 3 — Compare R3M vs SigLIP scores
print("Re-profiling with SigLIP for comparison...\n")

siglip_profiler = DatasetProfiler(embedding_model="siglip", device="cuda")
siglip_results = {}

for name, config in datasets.items():
    print(f"SigLIP profiling {name}...")
    start = time.time()
    siglip_profile = siglip_profiler.profile_from_hub(
        config["repo"],
        task_descriptions=config["tasks"],
        max_episodes=50,
    )
    siglip_elapsed = time.time() - start
    siglip_card = report_gen.generate(siglip_profile)
    siglip_results[name] = {
        "report_card": siglip_card,
        "time_seconds": siglip_elapsed,
    }

# Print comparison table
print(f"\n{'='*70}")
print(f"{'Dataset':<20} {'R3M Grade':>10} {'R3M Time':>10} {'SigLIP Grade':>12} {'SigLIP Time':>12}")
print(f"{'-'*70}")
for name in datasets:
    r3m_card = results[name]["report_card"]
    sig_card = siglip_results[name]["report_card"]
    r3m_t = results[name]["time_seconds"]
    sig_t = siglip_results[name]["time_seconds"]
    print(
        f"{name:<20} "
        f"{r3m_card.overall_grade:>5} ({r3m_card.overall_score:.2f}) "
        f"{r3m_t:>8.1f}s "
        f"{sig_card.overall_grade:>7} ({sig_card.overall_score:.2f}) "
        f"{sig_t:>9.1f}s"
    )

In [ ]:
# Cell 4 — Validate against ground truth
import json
from pathlib import Path
from scipy import stats
import numpy as np

# Ground truth success rates from published papers
ground_truth = {
    "pusht": 0.91,             # Diffusion Policy, Chi et al. RSS 2023
    "aloha_transfer": 0.82,    # ACT, Zhao et al. RSS 2023
    "aloha_insertion": 0.86,   # ACT, Zhao et al. RSS 2023
    "xarm_lift": 0.65,         # TD3+BC, LeRobot benchmark
}

# R3M coverage scores
r3m_scores = [
    results[name]["report_card"].coverage_score
    for name in ground_truth
]
gt_scores = list(ground_truth.values())

rho, p_value = stats.spearmanr(r3m_scores, gt_scores)
pearson_r, pearson_p = stats.pearsonr(r3m_scores, gt_scores)

print("R3M Coverage vs Ground Truth Success Rates")
print(f"{'='*50}")
print(f"Spearman rho: {rho:.3f} (p={p_value:.4f})")
print(f"Pearson r:    {pearson_r:.3f} (p={pearson_p:.4f})")
print()
print(f"{'Dataset':<20} {'R3M Coverage':>12} {'Ground Truth':>12}")
print(f"{'-'*50}")
for name in ground_truth:
    r3m_s = results[name]["report_card"].coverage_score
    gt_s = ground_truth[name]
    print(f"{name:<20} {r3m_s:>12.3f} {gt_s:>12.3f}")

In [ ]:
# Cell 5 — Save results
import json

output = {
    "validation_summary": {
        "spearman_rho": round(rho, 3),
        "spearman_p": round(p_value, 4),
        "pearson_r": round(pearson_r, 3),
        "pearson_p": round(pearson_p, 4),
    },
    "datasets": {},
}

for name in datasets:
    r3m_card = results[name]["report_card"]
    output["datasets"][name] = {
        "r3m_report_card": r3m_card.to_dict(),
        "r3m_time_seconds": round(results[name]["time_seconds"], 1),
        "ground_truth": ground_truth.get(name),
    }
    if name in siglip_results:
        sig_card = siglip_results[name]["report_card"]
        output["datasets"][name]["siglip_report_card"] = sig_card.to_dict()
        output["datasets"][name]["siglip_time_seconds"] = round(
            siglip_results[name]["time_seconds"], 1
        )

with open("r3m_validation_results.json", "w") as f:
    json.dump(output, f, indent=2)

print("Results saved to r3m_validation_results.json")
print(json.dumps(output["validation_summary"], indent=2))

In [ ]:
# Cell 6 — Timing comparison (single dataset)
import matplotlib.pyplot as plt

# Compare wall-clock time per dataset
dataset_names = list(datasets.keys())
r3m_times = [results[n]["time_seconds"] for n in dataset_names]
siglip_times = [siglip_results[n]["time_seconds"] for n in dataset_names]

x = np.arange(len(dataset_names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, r3m_times, width, label="R3M (ResNet-50)", color="#3498db")
bars2 = ax.bar(x + width/2, siglip_times, width, label="SigLIP (ViT-B/16)", color="#e74c3c")

ax.set_ylabel("Time (seconds)")
ax.set_title("Embedding Extraction Time: R3M vs SigLIP on T4 GPU")
ax.set_xticks(x)
ax.set_xticklabels(dataset_names, rotation=15)
ax.legend()
ax.grid(axis="y", alpha=0.3)

# Add time labels on bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{bar.get_height():.1f}s", ha="center", va="bottom", fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{bar.get_height():.1f}s", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig("r3m_vs_siglip_timing.png", dpi=150)
plt.show()

# Print speedup
avg_r3m = np.mean(r3m_times)
avg_siglip = np.mean(siglip_times)
print(f"\nAverage R3M time: {avg_r3m:.1f}s")
print(f"Average SigLIP time: {avg_siglip:.1f}s")
if avg_siglip > 0:
    speedup = avg_siglip / avg_r3m
    print(f"R3M is {speedup:.1f}x {'faster' if speedup > 1 else 'slower'} than SigLIP")